# La búsqueda del techo (v1)

Antes de comparar nada hay que elegir un escalar: hasta dónde sube la rampa del
término de adaptación de cada familia. Ese escalar lo eligen **mirando
resultados**, así que la cosa que lo elige es un experimento y necesita todo lo
que un experimento necesita — su propia escala, su propio rol del material, su
propia regla de desempate — y hasta ahora no tenía informe propio. Sus seis
celdas vivían adentro del reporte de campaña, que es el lugar donde se leen sus
consecuencias y no donde se juzga cómo se llegó a ellas.

Este cuaderno es ese informe. También corre el **ensayo local**, que es una cosa
distinta de la búsqueda y conviene no confundir:

| | escala | dónde escribe | su respuesta |
| --- | --- | --- | --- |
| **búsqueda** | 20 épocas · 3 semillas | `ceilings.json` | rige la campaña |
| **ensayo** | pocas épocas · 1 semilla | `ceilings.pilot.json` | **no se cita** |

El ensayo contesta *«¿el programa corre?»*, que es lo único que un ensayo puede
contestar. No contesta *«¿cuál es el techo?»*: la rampa avanza con la fracción de
entrenamiento transcurrida, así que a escala corta se satura en la segunda época
y todo techo se alcanza casi enseguida. Lo que mediría es un paisaje donde nada
más entrena.

Por eso son dos archivos y no uno. Con uno solo, el ensayo habría escrito donde
va la respuesta que la campaña consume, y una corrida completa lo habría gastado
sin una palabra.

In [1]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

repository: /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation


In [2]:
import json

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, harness, tables


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo."""
    display(Markdown(text))

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Qué registro rige ahora mismo

Leído del disco, no recordado. El registro completo le gana siempre al ensayo:
un ensayo no desplaza una medición.

In [3]:
proc = config.ceilings_provenance()
lineas = [f"**registro en vigor:** `{proc['source']}`"]
if proc["record"]:
    lineas.append(f"`{Path(proc['record']).name}` · "
                  f"{proc['epochs']} épocas · {proc['seeds']} semilla(s)")
    if proc.get("atRequiredScale") is False:
        lineas.append("**Está por debajo de la escala que el protocolo declara "
                      f"({proc['requiredScale']}): su techo no se cita.**")
else:
    lineas.append("ninguno todavía — ni la búsqueda ni el ensayo han corrido.")
show(" · ".join(lineas))

**registro en vigor:** `pilot` · `ceilings.pilot.json` · 3 épocas · 0 semilla(s) · **Está por debajo de la escala que el protocolo declara ({'epochs': 20, 'trials': 30}): su techo no se cita.**

## 2 · El ensayo local

Corre el mismo programa que la búsqueda, a su propia escala declarada, y escribe
a su propio archivo. Si ya existe un registro —el que sea— no vuelve a buscar: un
registro que existe significa que la búsqueda contestó, y volver a contestarla
porque un llamador posterior quería otra respuesta es exactamente el silencio que
la negativa de la campaña existe para impedir.

In [4]:
aforo = config.search_sizing()
bajo, alto = config.CEILING_RANGE
show(
    f"Motor: **{aforo['engine']}** con `GPSampler`.  \n"
    f"La búsqueda real: **{aforo['trials']} trials** por "
    f"`(familia, transferencia)` de **{aforo['epochs']} épocas** — "
    f"{aforo['families']} familias × {aforo['transfers']} transferencias × "
    f"{aforo['trials']} = **{aforo['runs']} corridas**.  \n"
    f"El ensayo: **{config.PILOT_SEARCH_TRIALS} trials** de "
    f"**{config.PILOT_SEARCH_EPOCHS} épocas**, que es "
    f"{config.PILOT_SEARCH_TRIALS * aforo['families'] * aforo['transfers']} "
    f"corridas.  \n"
    f"El rango es continuo, `[{bajo:g}, {alto:g}]` en escala logarítmica, y la "
    f"meseta la define la resolución del criterio sobre el rol de búsqueda: "
    f"**{config.SEARCH_RESOLUTION:g}**, que es una bolsa de las "
    f"{config.VALID_BAGS}."
)

Motor: **optuna** con `GPSampler`.  
La búsqueda real: **30 trials** por `(familia, transferencia)` de **20 épocas** — 2 familias × 6 transferencias × 30 = **360 corridas**.  
El ensayo: **4 trials** de **3 épocas**, que es 48 corridas.  
El rango es continuo, `[0.0001, 1]` en escala logarítmica, y la meseta la define la resolución del criterio sobre el rol de búsqueda: **0.05**, que es una bolsa de las 20.

In [5]:
# Descomentar para correrlo. Escribe `ceilings.pilot.json` y nada más.
# harness.run_search(pilot=True)

## 3 · Lo que la búsqueda eligió

Del registro en vigor. Si es el del ensayo, todo lo de abajo es plumbing y está
dicho arriba.

In [6]:
show(tables.objective("ceilings"))

> **Buscamos que el criterio se incline**, no un número en particular. El neutro es 1: un techo que aterriza ahí confirma la normalización por medición y no por argumento. Lo que invalida la elección es que la búsqueda no distinga —ahí el ganador lo pone la regla y no el criterio—, y la tabla lo dice en la columna que corresponda a su motor: una rejilla plana con semillas que no coinciden, o una meseta ancha. **Meseta uno significa que el criterio decidió.** Sobre el rango continuo la meseta la define la resolución del instrumento, 0.05, que es una bolsa de las 20 del rol de búsqueda: dos techos que difieren en menos que eso no son distinguibles por la medición.

In [7]:
registro = harness.search_record() or harness.search_record(pilot=True)
show(tables.render_ceilings(registro, markdown=True))

| Familia | Brazo | Transferencia | Techo | Criterio | Meseta | Trials |
|---|---|---|---|---|---|---|
| `creda` | `D` | M->S | **0.00282167** | 10.0 | 4 | 4 |
| `creda` | `D` | M->U | **0.000135666** | 80.0 | 4 | 4 |
| `creda` | `D` | S->M | **0.000165025** | 75.0 | 3 | 4 |
| `creda` | `D` | S->U | **0.00121795** | 50.0 | 4 | 4 |
| `creda` | `D` | U->M | **0.000302765** | 100.0 | 4 | 4 |
| `creda` | `D` | U->S | **0.000105352** | 20.0 | 4 | 4 |
| `milcreda` | `G` | M->S | **0.000643961** | 20.0 | 4 | 4 |
| `milcreda` | `G` | M->U | **0.00515442** | 80.0 | 2 | 4 |
| `milcreda` | `G` | S->M | **0.000558888** | 35.0 | 4 | 4 |
| `milcreda` | `G` | S->U | **0.000308476** | 35.0 | 4 | 4 |
| `milcreda` | `G` | U->M | **0.000582196** | 85.0 | 3 | 4 |
| `milcreda` | `G` | U->S | **0.0035048** | 15.0 | 4 | 4 |

En negrita el techo que puso la regla de meseta y no el criterio. **Meseta** es cuántos techos el ruido estimado no distinguió del mejor: uno significa que el criterio decidió.

In [8]:
show(tables.conclusion_ceilings(registro))

**creda** se queda en 0.000302765, elegido por una diferencia en el criterio sobre el rol `valid` con 4 trial(s) de 3 épocas y la resolución del criterio es 0.05, **por debajo de la escala que su respuesta necesita**. **milcreda** se queda en 0.000582196, elegido por una diferencia en el criterio sobre el rol `valid` con 4 trial(s) de 3 épocas y la resolución del criterio es 0.05, **por debajo de la escala que su respuesta necesita**.

### 3b · Qué techo rige en cada transferencia

La búsqueda mide unas pocas transferencias y las demás heredan. Esa herencia es
una aplicación **fuera de muestra** y se declara como tal.

In [9]:
show(tables.objective("ceilings.byTransfer"))

> **Buscamos saber cuántas de las 6 transferencias corren a un techo elegido mirándolas.** La búsqueda mide 6; las otras 0 heredan, y esa herencia es una aplicación fuera de muestra. Lo que hay que ver es si alguna de las medidas se aparta del ganador agrupado: si ninguna lo hace, separar las dos lecturas no cambió nada y la familia sigue corriendo a un coeficiente único; si alguna se aparta, deja de hacerlo y su promedio entre transferencias mezcla dos escalares.

In [10]:
transferencias = [f"{a}->{b}" for a, b in config.VERDICT_TRANSFERS]
show(tables.render_ceilings_by_transfer(registro, transferencias, markdown=True))

| Familia | M->U | U->M | M->S | S->M | U->S | S->U |
|---|---|---|---|---|---|---|
| `creda` | **0.000135666** | **0.000302765** | **0.00282167** | **0.000165025** | **0.000105352** | **0.00121795** |
| `milcreda` | **0.00515442** | **0.000582196** | **0.000643961** | **0.000558888** | **0.0035048** | **0.000308476** |

In [11]:
show(tables.conclusion_ceilings_by_transfer(registro, transferencias))

En las transferencias que la búsqueda midió rige el ganador de esa transferencia, por la misma lectura apareada y el mismo desempate. En las restantes rige el ganador de las medidas tomadas juntas: es una aplicación fuera de muestra y se declara como tal, porque ese escalar no se eligió mirándolas. **creda**: 6 medida(s), 0 heredada(s) a 0.000302765, y 5 de las medidas elige otro techo — `M->S` a 0.00282167, `M->U` a 0.000135666, `S->M` a 0.000165025, `S->U` a 0.00121795, `U->S` a 0.000105352. Ahí la familia deja de correr a un coeficiente único, así que su promedio entre transferencias mezcla dos escalares; dentro de cada transferencia todos los brazos siguen compartiendo el techo, que es lo que mantiene atribuible cada peldaño. **milcreda**: 6 medida(s), 0 heredada(s) a 0.000582196, y 5 de las medidas elige otro techo — `M->S` a 0.000643961, `M->U` a 0.00515442, `S->M` a 0.000558888, `S->U` a 0.000308476, `U->S` a 0.0035048. Ahí la familia deja de correr a un coeficiente único, así que su promedio entre transferencias mezcla dos escalares; dentro de cada transferencia todos los brazos siguen compartiendo el techo, que es lo que mantiene atribuible cada peldaño.

## 4 · Lo que esta búsqueda **no** midió

Es la limitación más grande del escalar que gobierna toda la campaña, y no se lee
en ninguna otra parte. La búsqueda midió unas transferencias y las otras heredaron
su techo sin que nadie lo comprobara ahí.

In [12]:
medidas = {f"{a}->{b}" for a, b in config.SEARCH_TRANSFERS}
todas = [f"{a}->{b}" for a, b in config.VERDICT_TRANSFERS]
heredadas = [t for t in todas if t not in medidas]
show(f"**Medidas:** {', '.join(sorted(medidas))} — "
     f"{len(medidas)} de {len(todas)}.  \n"
     f"**Heredadas:** {', '.join(heredadas)} — "
     f"{len(heredadas)} de {len(todas)}, fuera de muestra.")

**Medidas:** M->S, M->U, S->M, S->U, U->M, U->S — 6 de 6.  
**Heredadas:**  — 0 de 6, fuera de muestra.

## 5 · El sello


In [13]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())

SOURCES-SHA256 2900fdd65ba5f295f8badb5d982c0f194bc26f5ceef645837d52ea37e7551ede
